# The maths behind each slide

One section per slide of the migration deck. Each section has:

- **Theory and maths**: what the slide claims, written as equations.
- **Worked demo**: given code. Run it and read the output.
- **Your turn**: one short exercise (`# TODO`) with a check cell.
- **Interview link**: which question in the speaker notes this prepares you for.

| § | Slide | What you'll compute |
|---|---|---|
| 1 | The brief | How a PD becomes a grade and then capital, and why a tiny PD change can move capital |
| 2 | Replicate first | Why re-fitting gives different coefficients, and frozen parameters don't |
| 3 | Pipeline | A mini reproducible pipeline with run IDs and checksums |
| 4 | Reconciliation | Stage-by-stage comparison with tolerances, finding the first breaking stage |
| 5 | When numbers differ | Each SAS-vs-Python trap, reproduced in Python |
| 6 | Stepwise and VIF | SAS-style stepwise from scratch; VIF by hand; sign flips from collinearity |
| 7 | Monitoring | PSI, Gini from AUC, the Jeffreys test, and a RAG table |
| 8 | Plan | A parallel-run exit check |
| 9 | Closing | Capturing and verifying a golden benchmark |

Solutions are in the companion notebook. Same rules as before: write the TODOs yourself.

In [1]:
import json, hashlib
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (8, 4.5); plt.rcParams['axes.grid'] = True
pd.set_option('display.width', 120)

## Shared data (given)

A synthetic mortgage book. Compared with Day 1 there are two additions:
- `bti` (balance-to-income) is built to be **highly correlated with `dti`**. It has no effect of its own. It's there for the VIF section.
- `utilisation` has some **missing values**. They're there for the reconciliation and SAS-trap sections.

In [2]:
TRUE = {'const': -5.0, 'dti': 3.0, 'ltv': 1.5, 'months_in_arrears': 0.9, 'utilisation': 1.2}

def make_book(n, seed, shift=0.0):
    rng = np.random.default_rng(seed)
    dti = rng.normal(0.35 + shift, 0.12, n).clip(0, 1)
    df = pd.DataFrame({
        'account_id': np.arange(n),
        'dti': dti,
        'bti': (dti * 4 + rng.normal(0, 0.08, n)).clip(0, None),   # near-copy of dti
        'ltv': rng.normal(0.75 + shift, 0.15, n).clip(0.1, 1.2),
        'months_in_arrears': rng.poisson(0.3, n),
        'utilisation': rng.beta(2, 5, n),
        'noise': rng.normal(0, 1, n),
    })
    lin = TRUE['const'] + sum(TRUE[k] * df[k] for k in ['dti', 'ltv', 'months_in_arrears', 'utilisation'])
    df['default'] = rng.binomial(1, 1 / (1 + np.exp(-lin)))
    df.loc[rng.random(n) < 0.03, 'utilisation'] = np.nan          # 3% missing
    return df

dev = make_book(20000, seed=1)
print(dev.shape, f"default rate {dev['default'].mean():.2%}, missing utilisation {dev['utilisation'].isna().mean():.1%}")
print('corr(dti, bti) =', round(dev[['dti', 'bti']].corr().iloc[0, 1], 3))

(20000, 8) default rate 11.52%, missing utilisation 3.0%
corr(dti, bti) = 0.986


---
## 1. The brief: PD → grade → capital

**Claim on the slide:** "same PD, same grade" matters because **grade drives capital**.

**Maths.** A *master scale* maps a PD into a grade using fixed boundaries $b_1 < b_2 < \dots$:

$$\text{grade}(PD) = k \quad\text{if}\quad b_{k-1} \le PD < b_k$$

Capital is then calculated from the **grade's PD**, not the account's own PD. For retail mortgages, the IRB formula (Basel) for the capital requirement per £ of exposure is

$$K = LGD \cdot \left[\Phi\!\left(\frac{\Phi^{-1}(PD) + \sqrt{R}\,\Phi^{-1}(0.999)}{\sqrt{1-R}}\right) - PD\right], \qquad R = 0.15$$

and $RWA = 12.5 \times K \times EAD$.

In words: $\Phi^{-1}(PD)$ converts the PD to a z-score; adding $\sqrt{R}\,\Phi^{-1}(0.999)$ shifts it to a 1-in-1000-year bad economy; $\Phi$ converts back to a "stressed PD". Capital covers the gap between the stressed loss and the expected loss.

In [ ]:
BOUNDS = np.array([0.0005, 0.001, 0.0025, 0.005, 0.01, 0.025, 0.05, 0.10, 0.20])   # grade 1..10
GRADE_PD = np.array([0.0003, 0.0007, 0.0017, 0.0037, 0.0075, 0.017, 0.037, 0.075, 0.15, 0.35])

def to_grade(pd_):
    return np.searchsorted(BOUNDS, pd_, side='right') + 1    # 1..10

def irb_k(pd_, lgd=0.2, R=0.15):
    z = (stats.norm.ppf(pd_) + np.sqrt(R) * stats.norm.ppf(0.999)) / np.sqrt(1 - R)
    return lgd * (stats.norm.cdf(z) - pd_)

# Two accounts with PDs a hair either side of a boundary
for p in [0.0099999, 0.0100001]:
    g = to_grade(p)
    print(f'PD {p:.7f} -> grade {g} -> grade PD {GRADE_PD[g-1]:.4f} -> RWA per £100k = £{12.5 * irb_k(GRADE_PD[g-1]) * 100_000:,.0f}')

A PD difference of 0.0000002 moved the account up a grade and its RWA by thousands of pounds. That's why the reconciliation slide gives grade a **zero tolerance** and flags accounts near boundaries.

### Your turn 1.1
Write `near_boundary(pd_, tol)` returning a boolean array: True where a PD is within `tol` of **any** boundary in `BOUNDS`.

*Hint: `np.abs(pd_[:, None] - BOUNDS[None, :])` gives every account-to-boundary distance.*

In [ ]:
# TODO
def near_boundary(pd_, tol):
    ...

In [ ]:
# CHECK
test = np.array([0.0099999, 0.02, 0.0500004, 0.3])
assert list(near_boundary(test, 1e-6)) == [True, False, True, False]
print('1.1 passed')

**Interview link:** *"What about accounts sitting right on a grade boundary?"* (slide 5 notes).

---
## 2. Replicate first: why re-fitting is a model change

**Claim:** re-estimating in Python would give slightly different coefficients, so the migration freezes the SAS coefficients instead.

**Maths.** The MLE solves $\nabla\ell(\beta)=0$ *numerically*. Two things make re-fits differ:
1. **The optimiser and its stopping rule.** It stops when the change is below a tolerance, so different settings stop at slightly different $\hat\beta$.
2. **The data extract.** A handful of rows different (late-arriving data, a changed filter) moves the maximum.

Frozen coefficients avoid both: scoring becomes a deterministic calculation, $p = 1/(1+e^{-x^\top\beta_{frozen}})$.

In [ ]:
feats = ['dti', 'ltv', 'months_in_arrears', 'utilisation']
d = dev.dropna(subset=['utilisation'])
X_ = sm.add_constant(d[feats]); y_ = d['default']

sas_fit = sm.Logit(y_, X_).fit(disp=0, method='newton', tol=1e-10)        # stands in for "SAS"
loose = sm.Logit(y_, X_).fit(disp=0, method='bfgs', gtol=1e-3, maxiter=200)   # different optimiser, looser stop
drop = d.sample(frac=0.995, random_state=0)                                 # 0.5% of rows missing from the extract
extract = sm.Logit(drop['default'], sm.add_constant(drop[feats])).fit(disp=0)

comp = pd.DataFrame({'SAS (frozen)': sas_fit.params, 'different optimiser': loose.params, 'different extract': extract.params})
print(comp.round(5))

frozen = sas_fit.params
pd_sas = 1 / (1 + np.exp(-(X_ @ frozen)))
for name, b in [('different optimiser', loose.params), ('different extract', extract.params)]:
    pd_new = 1 / (1 + np.exp(-(X_ @ b)))
    flips = (to_grade(pd_sas.values) != to_grade(pd_new.values)).sum()
    print(f'{name:22s} max |PD diff| = {np.abs(pd_sas - pd_new).max():.2e}, grade flips = {flips}')

Both re-fits are "the same model" statistically, yet they move PDs and flip grades. Scoring with the frozen coefficients gives exactly the SAS PDs.

### Your turn 2.1
Write `score(df, params)` that returns PDs using **frozen** `params` (a Series including `'const'`). Do it with plain numpy/pandas, not `.predict()`, so you see the calculation.

In [ ]:
# TODO
def score(df, params):
    ...

In [ ]:
# CHECK
assert np.allclose(score(d, frozen), sas_fit.predict(X_))
print('2.1 passed')

**Interview link:** *"Will you re-fit the model in Python at all?"* (slide 3 notes).

---
## 3. The pipeline: every PD rebuildable from its run ID

**Claim:** each stage writes its output once, checksummed and tagged with a run ID; parameters live in config.

**The idea in code:**
- A **config** dict holds every parameter (coefficients, bin edges, grade boundaries).
- Each **stage** is a pure function: same input + same config gives the same output.
- A **checksum** (hash) of each stage output proves it hasn't changed. Change one parameter and only that stage and everything after it get a new hash.

In [ ]:
CONFIG = {
    'coefs': {k: float(v) for k, v in frozen.items()},
    'util_missing_fill': 0.0,          # how missing utilisation is treated (see section 5)
    'grade_bounds': BOUNDS.tolist(),
}

def checksum(df):
    return hashlib.sha256(pd.util.hash_pandas_object(df, index=True).values.tobytes()).hexdigest()[:12]

def st_cleanse(raw, cfg):
    out = raw[['account_id', 'dti', 'ltv', 'months_in_arrears', 'utilisation']].copy()
    out['utilisation'] = out['utilisation'].fillna(cfg['util_missing_fill'])
    return out

def st_score(clean, cfg):
    c = cfg['coefs']
    eta = c['const'] + sum(c[f] * clean[f] for f in c if f != 'const')
    return pd.DataFrame({'account_id': clean['account_id'], 'eta': eta})

def st_pd(scored, cfg):
    return pd.DataFrame({'account_id': scored['account_id'], 'pd': 1 / (1 + np.exp(-scored['eta']))})

def st_grade(pds, cfg):
    g = np.searchsorted(np.array(cfg['grade_bounds']), pds['pd'], side='right') + 1
    return pd.DataFrame({'account_id': pds['account_id'], 'pd': pds['pd'], 'grade': g})

STAGES = [('cleansed', st_cleanse), ('score', st_score), ('pd', st_pd), ('grade', st_grade)]

def run_pipeline(raw, cfg, run_month):
    run_id = f"{run_month}-{hashlib.sha256(json.dumps(cfg, sort_keys=True).encode()).hexdigest()[:8]}"
    outputs, manifest, x = {}, {'run_id': run_id, 'raw': checksum(raw)}, raw
    for name, fn in STAGES:
        x = fn(x, cfg)
        outputs[name] = x
        manifest[name] = checksum(x)
    return outputs, manifest

out1, man1 = run_pipeline(dev, CONFIG, '2026-10')
out2, man2 = run_pipeline(dev, CONFIG, '2026-10')
print(json.dumps(man1, indent=1))
print('Rerun identical:', man1 == man2)

### Your turn 3.1
Make `cfg2`: a **deep copy** of `CONFIG` with the `ltv` coefficient increased by 0.01. Run the pipeline with it into `man3`, then build `changed`: the list of stage names (from `['raw', 'cleansed', 'score', 'pd', 'grade']`) whose checksum differs from `man1`.

*Hint: `json.loads(json.dumps(CONFIG))` is a quick deep copy.*

In [ ]:
# TODO
cfg2 = ...
_, man3 = ...
changed = ...
print(changed)

In [ ]:
# CHECK
assert changed == ['score', 'pd', 'grade']
assert man3['run_id'] != man1['run_id']
print('3.1 passed: the change shows up exactly where it should, and gets its own run ID')

**Interview link:** *"Why store every intermediate stage?"* and *"What happens when upstream data gets restated?"* (slide 4 notes).

---
## 4. Reconciliation: every stage, with tolerances

**Claim:** compare every stage, each to its own tolerance, and find the **first** stage that breaks.

**Maths.** For numeric outputs $a$ (SAS) and $b$ (Python), an account passes if

$$|a - b| \le \text{atol} \quad\text{(absolute)} \qquad\text{or}\qquad \frac{|a-b|}{|a|} \le \text{rtol} \quad\text{(relative)}$$

Categorical outputs (bins, grades) must match exactly.

Below, a simulated "SAS" run differs from Python in two ways: **floating-point noise** everywhere (harmless), and the **missing-utilisation trap** from section 5 (a real logic break). Watch the reconciliation tell them apart.

In [ ]:
py = run_pipeline(dev, CONFIG, '2026-10')[0]

# Simulated SAS outputs: same logic, but SAS treated missing utilisation differently
cfg_sas = json.loads(json.dumps(CONFIG)); cfg_sas['util_missing_fill'] = 0.0
raw_sas = dev.copy()
raw_sas['utilisation'] = raw_sas['utilisation'].fillna(raw_sas['utilisation'].median())   # an undocumented SAS step
sas = run_pipeline(raw_sas, cfg_sas, '2026-10')[0]
rng = np.random.default_rng(7)
sas['score'] = sas['score'].assign(eta=sas['score']['eta'] * (1 + rng.normal(0, 1e-14, len(dev))))  # float noise
sas['pd'] = sas['pd'].assign(pd=sas['pd']['pd'] * (1 + rng.normal(0, 1e-14, len(dev))))

TOL = {'cleansed': ('utilisation', 'rtol', 1e-9), 'score': ('eta', 'atol', 1e-8), 'pd': ('pd', 'atol', 1e-6), 'grade': ('grade', 'exact', 0)}

def reconcile(sas, py):
    rows = []
    for stage, (col, kind, tol) in TOL.items():
        a, b = sas[stage][col].values, py[stage][col].values
        if kind == 'exact':
            bad = a != b
        elif kind == 'atol':
            bad = np.abs(a - b) > tol
        else:
            bad = np.abs(a - b) > tol * np.abs(a).clip(1e-300)
        rows.append({'stage': stage, 'column': col, 'rule': f'{kind} {tol:g}', 'breaks': int(bad.sum()),
                     'max_abs_diff': float(np.abs(a - b).max()) if kind != 'exact' else float(bad.sum())})
    return pd.DataFrame(rows)

recon = reconcile(sas, py)
recon

The float noise (~1e-14) passes every tolerance. The missing-value difference breaks at **`cleansed`**, the first stage, and cascades downstream. Without stage-by-stage outputs, you'd only see the PD and grade breaks and have to guess where they came from.

### Your turn 4.1
1. `first_break`: the name of the first stage in `recon` with breaks > 0.
2. `culprits`: the `account_id`s that break at that stage (compare `sas[first_break]` and `py[first_break]` on `utilisation`).
3. Confirm they're exactly the accounts with missing utilisation in `dev`.

In [ ]:
# TODO
first_break = ...
culprits = ...
print(first_break, len(culprits))

In [ ]:
# CHECK
assert first_break == 'cleansed'
assert set(culprits) == set(dev.loc[dev['utilisation'].isna(), 'account_id'])
print('4.1 passed: root cause located to one stage, one column, and a precise set of accounts')

**Interview link:** *"How do you find the root cause quickly?"* and *"Why reconcile derived variables separately if the PD matches?"* (slides 5 and 6 notes).

---
## 5. When the numbers differ: the SAS traps, reproduced

Each cell shows one row of the slide's table.

### 5a. Missing values
SAS: missing is **smaller than any number**. Python: comparisons with `NaN` are **always False**.

In [ ]:
dpd = pd.Series([0, 45, np.nan, 120])
python_flag = dpd < 30                                   # NaN < 30 -> False
sas_flag = dpd.fillna(-np.inf) < 30                      # emulate SAS: missing < anything -> True
print(pd.DataFrame({'dpd': dpd, 'python: dpd<30': python_flag, 'SAS: dpd<30': sas_flag}))

### 5b. Rounding
SAS `ROUND` sends halves **away from zero**. Python's `round()` and `np.round` send halves **to even**, and floating-point storage makes "halves" unreliable.

In [ ]:
for v in [0.5, 1.5, 2.5, -2.5]:
    print(f'{v:5}: python round -> {round(v)},  numpy -> {np.round(v)}')
print('0.047125 to 5 dp, python:', round(0.047125, 5), '  (stored as', f'{0.047125:.20f})')

### Your turn 5.1
Write `sas_round(x, decimals)` that rounds halves **away from zero**, using the `decimal` module so binary storage doesn't interfere.

*Hint: `Decimal(str(x)).quantize(Decimal('1e-5'), rounding=ROUND_HALF_UP)`. In `decimal`, `ROUND_HALF_UP` means away from zero.*

In [ ]:
from decimal import Decimal, ROUND_HALF_UP
# TODO
def sas_round(x, decimals=0):
    ...

In [ ]:
# CHECK
assert sas_round(2.5) == 3 and sas_round(-2.5) == -3 and sas_round(0.5) == 1
assert sas_round(0.047125, 5) == 0.04713
print('5.1 passed')

### 5c. Bin edges
SAS format `low-<10` means **[low, 10)**: 10 itself goes into the next bin. `pd.cut` defaults to **right-inclusive** (low, 10].

In [ ]:
vals = pd.Series([9.99, 10.0, 10.01])
print(pd.DataFrame({'value': vals,
                    'pd.cut default': pd.cut(vals, [-np.inf, 10, np.inf]),
                    'SAS low-<10 (right=False)': pd.cut(vals, [-np.inf, 10, np.inf], right=False)}))

### 5d. Dates
SAS stores dates as **days since 1 January 1960**.

In [ ]:
sas_dates = pd.Series([0, 23741, 24410])            # numbers as they'd arrive from a SAS dataset
print(pd.to_datetime(sas_dates, unit='D', origin='1960-01-01'))
print('Wrong (1970 epoch):', pd.to_datetime(23741, unit='D').date())

### 5e. Event order in PROC LOGISTIC
By default SAS models $P(Y = 0)$, the lower ordered value, unless the code says `DESCENDING` or `EVENT='1'`. Modelling $1-y$ flips every sign.

In [ ]:
m1 = sm.Logit(d['default'], X_).fit(disp=0)
m0 = sm.Logit(1 - d['default'], X_).fit(disp=0)
print(pd.DataFrame({'models P(default)': m1.params, 'models P(no default) [SAS default]': m0.params}).round(4))

**Interview link:** every row of slide 6, plus *"Give an example of a numerical rather than logic break."*

---
## 6. Variable selection: SAS-style stepwise and VIF

### 6a. The tests stepwise uses
For candidate variable $j$ (one coefficient):

- **Score test (entry).** Fit the model *without* $j$, giving fitted PDs $\tilde p$. With the design matrix $X$ *including* $j$:
$$U = X^\top(y - \tilde p), \quad I = X^\top \tilde W X, \quad S = U^\top I^{-1} U \sim \chi^2_1$$
- **Wald test (removal).** From the model *with* $j$: $W = (\hat\beta_j / SE_j)^2 \sim \chi^2_1$.

SAS defaults: enter if $p < 0.05$ (SLENTRY), remove if $p > 0.05$ (SLSTAY). Cut-off $\chi^2_1 = 3.84 = 1.96^2$.

In [ ]:
def score_test(df, selected, cand):
    yv = df['default'].values
    Xr = sm.add_constant(df[selected]) if selected else pd.DataFrame({'const': np.ones(len(df))}, index=df.index)
    p0 = np.asarray(sm.Logit(yv, Xr).fit(disp=0).predict(Xr))
    Xf = np.column_stack([Xr.values, df[cand].values])
    U = Xf.T @ (yv - p0)
    I = Xf.T @ (Xf * (p0 * (1 - p0))[:, None])
    S = float(U @ np.linalg.solve(I, U))
    return S, stats.chi2.sf(S, 1)

ds = dev.dropna(subset=['utilisation'])
for c in ['dti', 'bti', 'noise']:
    print(c, 'entering an empty model: S = %.1f, p = %.3g' % score_test(ds, [], c))

### Your turn 6.1: SAS-style stepwise
Write `stepwise(df, candidates, slentry=0.05, slstay=0.05)` returning the list of selected variables, in the order they entered. Each loop:
1. **Entry:** score-test every candidate not yet selected. If the smallest p < `slentry`, add that variable.
2. **Removal:** fit the model with the selected variables. If the largest Wald p (excluding `const`) > `slstay`, remove that variable.
3. Stop when a full loop neither adds nor removes. (Add a max-iterations guard.)

In [ ]:
CANDS = ['dti', 'bti', 'ltv', 'months_in_arrears', 'utilisation', 'noise']
# TODO
def stepwise(df, candidates, slentry=0.05, slstay=0.05):
    ...

chosen = stepwise(ds, CANDS)
print(chosen)

In [ ]:
# CHECK
assert set(['dti', 'ltv', 'months_in_arrears', 'utilisation']) <= set(chosen) or set(['bti', 'ltv', 'months_in_arrears', 'utilisation']) <= set(chosen)
assert 'noise' not in chosen
assert not ({'dti', 'bti'} <= set(chosen)), 'collinear pair should not both survive here'
print('6.1 passed')

### 6b. VIF by hand
For variable $j$, regress it on all the other predictors and take that $R^2_j$:

$$VIF_j = \frac{1}{1 - R_j^2} \qquad\Rightarrow\qquad SE_j \text{ is inflated by } \sqrt{VIF_j}$$

It uses **predictors only**, never the default flag.

In [ ]:
def vif(df, cols):
    out = {}
    for j in cols:
        others = [c for c in cols if c != j]
        r2 = sm.OLS(df[j], sm.add_constant(df[others])).fit().rsquared
        out[j] = 1 / (1 - r2)
    return pd.Series(out)

v = vif(ds, ['dti', 'bti', 'ltv', 'months_in_arrears', 'utilisation'])
print(v.round(2))
print('\nSE inflation factor sqrt(VIF):'); print(np.sqrt(v).round(2))

### 6c. What breaks if you ignore it
Fit a model with **both** `dti` and `bti` on 200 bootstrap samples. Both describe the same thing, so the model can't tell them apart.

In [ ]:
rng = np.random.default_rng(3)
small = ds.sample(3000, random_state=0)
rows = []
for b in range(200):
    s_ = small.sample(len(small), replace=True, random_state=int(rng.integers(1e9)))
    for spec, cols in [('dti only', ['dti', 'ltv', 'months_in_arrears', 'utilisation']),
                       ('dti + bti', ['dti', 'bti', 'ltv', 'months_in_arrears', 'utilisation'])]:
        f = sm.Logit(s_['default'], sm.add_constant(s_[cols])).fit(disp=0)
        rows.append({'spec': spec, 'dti_coef': f.params['dti']})
boot = pd.DataFrame(rows)
fig, ax = plt.subplots()
for spec, g in boot.groupby('spec'):
    ax.hist(g['dti_coef'], bins=30, alpha=0.6, label=spec)
ax.axvline(0, color='black', ls='--'); ax.set_xlabel('dti coefficient across bootstrap samples'); ax.legend()
ax.set_title('Collinearity: same data, unstable coefficient'); plt.show()
print(boot.groupby('spec')['dti_coef'].agg(['mean', 'std', lambda x: (x < 0).mean()]).rename(columns={'<lambda_0>': 'share negative'}).round(3))

### Your turn 6.2
Compute `ratio`: the standard deviation of the `dti` coefficient in the `'dti + bti'` spec divided by the `'dti only'` spec. Compare it with $\sqrt{VIF_{dti}}$ from 6b. They should be in the same ballpark. That's VIF measuring exactly what it claims to.

In [ ]:
# TODO
ratio = ...
print(ratio, np.sqrt(v['dti']))

In [ ]:
# CHECK
assert ratio > 2
assert 0.4 < ratio / np.sqrt(v['dti']) < 2.5
print('6.2 passed')

### 6d. Why stepwise p-values are optimistic
Testing $k$ pure-noise candidates at 5%: $P(\text{at least one enters}) = 1 - 0.95^k$.

In [ ]:
for k in [1, 5, 10, 30, 50]:
    print(f'{k:3d} noise candidates -> {1 - 0.95**k:.0%} chance at least one gets in')

**Interview link:** all eight questions in the slide 7 notes.

---
## 7. Monitoring: PSI, Gini and the Jeffreys test

We score the development book and a **drifted** live book (`shift=0.05`: higher DTI and LTV).

### 7a. PSI
Bin by **development** deciles. With $e_i$ = development share, $a_i$ = live share in bin $i$:

$$PSI = \sum_i (a_i - e_i)\,\ln\frac{a_i}{e_i}$$

Rule of thumb: < 0.10 stable, 0.10–0.25 investigate, > 0.25 significant shift.

In [ ]:
live = make_book(20000, seed=99, shift=0.05)
dev_pd = score(dev.fillna({'utilisation': 0}), frozen)
live_pd = score(live.fillna({'utilisation': 0}), frozen)

def psi(expected, actual, bins=10):
    edges = np.unique(np.quantile(expected, np.linspace(0, 1, bins + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    e = np.histogram(expected, edges)[0] / len(expected)
    a = np.histogram(actual, edges)[0] / len(actual)
    e, a = np.clip(e, 1e-6, None), np.clip(a, 1e-6, None)
    return float(np.sum((a - e) * np.log(a / e)))

print('Score PSI:', round(psi(dev_pd, live_pd), 4))
print('Same-population PSI (should be ~0):', round(psi(dev_pd, score(make_book(20000, seed=5).fillna({'utilisation': 0}), frozen)), 4))

### Your turn 7.1
Build `input_psi`: a Series of PSI for each input in `['dti', 'ltv', 'months_in_arrears', 'utilisation']`, development vs live (drop missing values first). Then `worst = input_psi.idxmax()`.

Note: `months_in_arrears` has few distinct values, so `np.unique` in `psi` merges duplicate edges. That's intended.

In [ ]:
# TODO
input_psi = ...
worst = ...
print(input_psi.round(4), worst)

In [ ]:
# CHECK
assert worst in ('dti', 'ltv')
assert input_psi['utilisation'] < 0.02
print('7.1 passed: the drift is traced to the inputs that actually shifted')

### 7b. Gini from AUC
AUC is the probability that a random defaulter has a higher PD than a random non-defaulter. With ranks $r_i$ of all PDs and $n_1$ defaulters, $n_0$ non-defaulters:

$$AUC = \frac{\sum_{i \in \text{defaulters}} r_i - n_1(n_1+1)/2}{n_1 n_0}, \qquad Gini = 2\,AUC - 1$$

In [ ]:
def gini(y, p):
    r = stats.rankdata(p)
    n1 = y.sum(); n0 = len(y) - n1
    auc = (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)
    return 2 * auc - 1

print('Gini dev :', round(gini(dev['default'].values, dev_pd.values), 4))
print('Gini live:', round(gini(live['default'].values, live_pd.values), 4))

### 7c. The Jeffreys test (calibration per grade)
For a grade with $N$ accounts, $D$ defaults and grade PD $p$: give the true default rate a $\text{Beta}(D + \tfrac12,\; N - D + \tfrac12)$ distribution. The p-value is

$$p\text{-value} = P\big(\text{Beta}(D + \tfrac12, N - D + \tfrac12) \le p\big)$$

the chance the true default rate is at or below the PD. **Small** means the PD looks **too low**.

In [ ]:
def jeffreys(N, D, grade_pd):
    return stats.beta.cdf(grade_pd, D + 0.5, N - D + 0.5)

t = pd.DataFrame({'grade': to_grade(live_pd.values), 'default': live['default'].values, 'pd': live_pd.values})
by_grade = t.groupby('grade').agg(N=('default', 'size'), D=('default', 'sum'), avg_pd=('pd', 'mean'))
by_grade['grade_pd'] = GRADE_PD[by_grade.index - 1]
by_grade['observed_dr'] = by_grade['D'] / by_grade['N']

### Your turn 7.2
Add a column `jeffreys_p` to `by_grade` (use `grade_pd`), and a column `flag` that is `'Red'` if `jeffreys_p < 0.01`, `'Amber'` if `< 0.05`, else `'Green'`.

In [ ]:
# TODO

by_grade.round(4)

In [ ]:
# CHECK
assert {'jeffreys_p', 'flag'} <= set(by_grade.columns)
assert by_grade['jeffreys_p'].between(0, 1).all()
print('7.2 passed'); print(by_grade['flag'].value_counts())

### 7d. The one-page RAG table
Everything above, turned into the slide's table.

In [ ]:
def rag(value, amber, red, higher_is_worse=True):
    if not higher_is_worse:
        value, amber, red = -value, -amber, -red
    return 'Red' if value >= red else 'Amber' if value >= amber else 'Green'

g_dev, g_live = gini(dev['default'].values, dev_pd.values), gini(live['default'].values, live_pd.values)
page = pd.DataFrame([
    {'check': 'Score PSI', 'signal': 'Leading', 'value': psi(dev_pd, live_pd), 'status': rag(psi(dev_pd, live_pd), 0.10, 0.25)},
    {'check': f'Worst input PSI ({worst})', 'signal': 'Leading', 'value': input_psi.max(), 'status': rag(input_psi.max(), 0.10, 0.25)},
    {'check': 'Missing rate: utilisation', 'signal': 'Leading', 'value': live['utilisation'].isna().mean(), 'status': rag(live['utilisation'].isna().mean(), 0.05, 0.10)},
    {'check': 'Gini relative drop', 'signal': 'Lagging', 'value': 1 - g_live / g_dev, 'status': rag(1 - g_live / g_dev, 0.05, 0.10)},
    {'check': 'Grades failing Jeffreys (p<0.05)', 'signal': 'Lagging', 'value': int((by_grade['jeffreys_p'] < 0.05).sum()), 'status': rag((by_grade['jeffreys_p'] < 0.05).sum(), 1, 3)},
])
page.round(4)

**Interview link:** every question in the slide 8 notes. Thresholds here are illustrative, as on the slide.

---
## 8. Plan: the parallel-run exit check

**Claim:** cut over only after three consecutive clean months, not on a fixed date.

### Your turn 8.1
`monthly` below records, for each parallel-run month, the number of reconciliation breaks per stage. Write `ready_to_cutover(monthly, n=3)` that returns True only if the **last `n` months** all have zero breaks in every stage.

In [ ]:
monthly = pd.DataFrame({
    'month':   ['2028-01', '2028-02', '2028-03', '2028-04', '2028-05'],
    'cleansed': [0, 2, 0, 0, 0],
    'pd':       [0, 2, 0, 0, 0],
    'grade':    [1, 1, 0, 0, 0],
})
# TODO
def ready_to_cutover(monthly, n=3):
    ...

In [ ]:
# CHECK
assert ready_to_cutover(monthly) is True
assert ready_to_cutover(monthly.iloc[:4]) is False
print('8.1 passed')

**Interview link:** *"Why three months of parallel running?"* (slide 9 notes).

---
## 9. Closing: capture and verify the golden benchmark

**Claim:** while both systems exist, capture SAS outputs at every stage, with a manifest, so equivalence can be proven later.

The manifest records a checksum per stage. Later, anyone can recompute the checksums from the archived files and prove nothing was altered.

In [ ]:
import os, tempfile
archive = tempfile.mkdtemp()
manifest = {'system': 'SAS (simulated)', 'run_month': '2026-10', 'stages': {}}
for stage, frame in sas.items():
    path = os.path.join(archive, f'{stage}.csv')
    frame.to_csv(path, index=False, float_format='%.17g')       # full precision
    manifest['stages'][stage] = {'file': f'{stage}.csv', 'rows': len(frame),
                                 'sha256': hashlib.sha256(open(path, 'rb').read()).hexdigest()}
json.dump(manifest, open(os.path.join(archive, 'manifest.json'), 'w'), indent=1)
print(json.dumps(manifest, indent=1)[:600], '...')

### Your turn 9.1
Write `verify_archive(folder)` that reads `manifest.json`, recomputes each file's sha256, and returns a dict `{stage: True/False}`. Then tamper with one file and show verification catches it.

In [ ]:
# TODO
def verify_archive(folder):
    ...

print(verify_archive(archive))

In [ ]:
# CHECK
assert all(verify_archive(archive).values())
with open(os.path.join(archive, 'pd.csv'), 'a') as f:
    f.write('999999,0.5\n')                                          # tamper
res = verify_archive(archive)
assert res['pd'] is False and res['grade'] is True
print('9.1 passed: the archive proves itself, and catches any change')

**Interview link:** *\"What exactly would you archive?\"* (slide 10 notes).

---
**Done?** For each section, say out loud the matching interview answer from the speaker notes, using the numbers you just produced. If an answer still feels like something you'd recite rather than explain, tell me which, and we'll drill it.